In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('preprocessed_data.csv')


In [3]:
df.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'revol_util', 'total_acc',
       'loan_status', 'fico_score'],
      dtype='object')

In [4]:
df.shape

(1345310, 17)

In [5]:
df.isnull().sum()

loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
dti                    0
delinq_2yrs            0
inq_last_6mths         0
open_acc               0
revol_util             0
total_acc              0
loan_status            0
fico_score             0
dtype: int64

In [6]:
median_inc = df[df['annual_inc'] > 0]['annual_inc'].median()
df['annual_inc'] = df['annual_inc'].replace(0, median_inc)

In [7]:
# Most important ones for loan default
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']
df['installment_to_income'] = df['installment'] / df['annual_inc']
df['loan_to_installment'] = df['loan_amnt'] / df['installment']
df['dti_to_income'] = df['dti'] * df['annual_inc']  # absolute debt burden

In [8]:
df['fico_category'] = pd.cut(df['fico_score'], 
                              bins=[300, 580, 670, 740, 800, 850],
                              labels=['Poor', 'Fair', 'Good', 'VeryGood', 'Exceptional'])

# FICO vs Interest Rate mismatch (risky if high rate despite good score)
df['fico_rate_mismatch'] = df['int_rate'] / df['fico_score']

In [9]:
# Inquiries to open accounts ratio (desperation signal)
df['inq_to_open_acc'] = df['inq_last_6mths'] / (df['open_acc'] + 1)

# Delinquency rate
df['delinq_rate'] = df['delinq_2yrs'] / (df['total_acc'] + 1)

# Credit utilization proxy
df['acc_utilization'] = df['open_acc'] / (df['total_acc'] + 1)

# Revolving utilization risk flag
df['high_revol_util'] = (df['revol_util'] > 80).astype(int)

In [10]:
# High interest rate is itself a default signal
df['high_int_rate'] = (df['int_rate'] > 15).astype(int)

# Interest rate vs grade alignment (mismatches are risky)
grade_avg_rate = df.groupby('grade')['int_rate'].transform('mean')
df['rate_vs_grade_avg'] = df['int_rate'] - grade_avg_rate

In [11]:
df['emp_unknown'] = (df['emp_length'] == -1).astype(int)

# Step 2 - Replace -1 with median for calculation
median_emp = df[df['emp_length'] != -1]['emp_length'].median()
print(f"Median emp_length: {median_emp}")  # Check what median is

emp_length_calc = df['emp_length'].replace(-1, median_emp)

# Step 3 - Recreate derived features safely
df['income_per_emp_year'] = df['annual_inc'] / (emp_length_calc + 1)
df['stable_employment'] = (emp_length_calc >= 5).astype(int)

# Verify
print(df['income_per_emp_year'].isna().sum()) 

Median emp_length: 6.0
0


In [12]:
df['risk_score'] = (
    df['dti'] * 0.3 +
    df['int_rate'] * 0.3 +
    df['inq_last_6mths'] * 0.2 +
    df['delinq_2yrs'] * 0.2
)

# High risk flag
df['high_risk'] = (
    (df['dti'] > 30) &
    (df['int_rate'] > 15) &
    (df['delinq_2yrs'] > 0)
).astype(int)

In [13]:
# Convert term to numeric
# Long term + high rate = risky combo
df['term_rate_interaction'] = df['term'] * df['int_rate']

In [14]:
# Correlation of new features with target
new_features = ['loan_to_income', 'installment_to_income', 
                 'fico_rate_mismatch', 'risk_score', 'high_risk']

df[new_features + ['loan_status']].corr()['loan_status'].sort_values()

loan_to_income          -0.000281
installment_to_income   -0.000279
high_risk                0.044636
risk_score               0.173548
fico_rate_mismatch       0.260436
loan_status              1.000000
Name: loan_status, dtype: float64

In [15]:
import numpy as np

df['log_annual_inc'] = np.log1p(df['annual_inc'])
df['log_loan_amnt'] = np.log1p(df['loan_amnt'])

# Recreate ratios with log values
df['loan_to_income'] = df['log_loan_amnt'] / df['log_annual_inc']
df['installment_to_income'] = df['installment'] / df['log_annual_inc']

# Recheck correlation
print(df[['loan_to_income', 'installment_to_income', 'loan_status']].corr()['loan_status'])

loan_to_income           0.102497
installment_to_income    0.059568
loan_status              1.000000
Name: loan_status, dtype: float64


In [16]:
from sklearn.model_selection import train_test_split

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1076248, 36)
(269062, 36)


In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X_train['fico_category'] = le.fit_transform(X_train['fico_category'].astype(str))
X_test['fico_category'] = le.transform(X_test['fico_category'].astype(str))

# Verify
print(X_train['fico_category'].unique())  # Should be numbers now
print(X_train.dtypes[X_train.dtypes == 'object'])

[2 3 1 0]
Series([], dtype: object)


In [18]:
from imblearn.over_sampling import SMOTE

# Keep imbalance realistic (do NOT use 1.0 immediately)
smote = SMOTE(
    sampling_strategy=0.6,   # minority will be 60% of majority
    random_state=42
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print("X_train_sm shape:", X_train_sm.shape)
print("y_train_sm value counts:\n", y_train_sm.value_counts())

After SMOTE:
X_train_sm shape: (1378241, 36)
y_train_sm value counts:
 loan_status
0    861401
1    516840
Name: count, dtype: int64


In [19]:
sample_size = 400000

X_train_sample = X_train_sm.sample(n=sample_size, random_state=42)
y_train_sample = y_train_sm.loc[X_train_sample.index]

In [20]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

param_grid = {
    "n_estimators": [300,400,500],
    "max_depth": [4,5,6,7],
    "learning_rate": [0.03,0.05,0.07],
    "subsample": [0.7,0.8,1],
    "colsample_bytree": [0.7,0.8,1],
    "scale_pos_weight": [2,3,4]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='recall',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [21]:
random_search.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_cons...
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1],
                                        'learning_rate': [0.03, 0.05, 0.07],
                                        'max_depth': [4, 5, 6, 7],
                                        'n_estimators': [300, 400, 500],
                                        'scale_pos_weight': [2, 3, 4],
                                        'subsample': [0.7, 0.8, 1]},
                   random_state=42, scoring='recall', verbose=2)

In [22]:
best_params = random_search.best_params_

final_model = XGBClassifier(**best_params)

final_model.fit(X_train_sm, y_train_sm)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
from sklearn.metrics import classification_report, roc_auc_score
pred = final_model.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.89      0.60      0.72    215350
           1       0.31      0.71      0.43     53712

    accuracy                           0.62    269062
   macro avg       0.60      0.66      0.57    269062
weighted avg       0.78      0.62      0.66    269062



In [24]:
from sklearn.metrics import classification_report
import numpy as np

probs = final_model.predict_proba(X_test)[:,1]

for t in np.arange(0.4, 0.7, 0.05):

    print(f"\nThreshold = {t:.2f}")
    preds = (probs >= t).astype(int)
    print(classification_report(y_test, preds))


Threshold = 0.40
              precision    recall  f1-score   support

           0       0.92      0.41      0.57    215350
           1       0.27      0.86      0.41     53712

    accuracy                           0.50    269062
   macro avg       0.59      0.63      0.49    269062
weighted avg       0.79      0.50      0.53    269062


Threshold = 0.45
              precision    recall  f1-score   support

           0       0.91      0.50      0.65    215350
           1       0.28      0.79      0.42     53712

    accuracy                           0.56    269062
   macro avg       0.60      0.65      0.53    269062
weighted avg       0.78      0.56      0.60    269062


Threshold = 0.50
              precision    recall  f1-score   support

           0       0.89      0.60      0.72    215350
           1       0.31      0.71      0.43     53712

    accuracy                           0.62    269062
   macro avg       0.60      0.66      0.57    269062
weighted avg       0

In [25]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, probs)

print("PR-AUC:", pr_auc)

PR-AUC: 0.3807369061474013


In [26]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, probs)

print("PR-AUC:", pr_auc)

PR-AUC: 0.3807369061474013


In [27]:
from sklearn.metrics import roc_auc_score, average_precision_score

# You already have probs from previous cell
probs = final_model.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test, probs)
auc_pr = average_precision_score(y_test, probs)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")

AUC-ROC: 0.7155
AUC-PR:  0.3807


In [28]:
best_params

{'subsample': 0.8,
 'scale_pos_weight': 4,
 'n_estimators': 400,
 'max_depth': 6,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

In [29]:
from sklearn.metrics import accuracy_score

# Using threshold 0.55
y_pred = (probs >= 0.55).astype(int)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.6768


In [31]:
df.to_csv('feature_engineered_data.csv',index=False)

In [ ]:
from catboost import CatBoostClassifier

 # your ratio from before

model_cat = CatBoostClassifier(
    auto_class_weights='Balanced',  # No need for ratio at all!
    random_state=42,
    verbose=100,
    eval_metric='F1'
)

model_cat.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_proba_cat = model_cat.predict_proba(X_test)[:, 1]

for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    y_pred = (y_proba_cat >= threshold).astype(int)
    print(f"\nThreshold = {threshold}")
    print(classification_report(y_test, y_pred))

In [ ]:
from lightgbm import LGBMClassifier

model_lgbm = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model_lgbm.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_proba_cat = model_lgbm.predict_proba(X_test)[:, 1]

for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    y_pred = (y_proba_cat >= threshold).astype(int)
    print(f"\nThreshold = {threshold}")
    print(classification_report(y_test, y_pred))

In [ ]:
# Use 0.001 instead of 0.01
top_features = importance[importance > 0.001].index.tolist()
print(f"Number of features selected: {len(top_features)}")
print(top_features)

X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

In [ ]:
X_train_selected

In [ ]:
import matplotlib.pyplot as plt

importance.sort_values(ascending=True).plot(kind='barh', figsize=(10,12))
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg / pos

In [ ]:
# Drop very low importance features
drop_features = ['installment', 'dti_to_income', 'open_acc', 
                 'delinq_2yrs', 'installment_to_income', 
                 'delinq_rate', 'revol_util', 'loan_amnt',
                 'dti', 'rate_vs_grade_avg']

X_train_final = X_train_selected.drop(drop_features, axis=1)
X_test_final = X_test_selected.drop(drop_features, axis=1)

print(f"Final features: {X_train_final.shape[1]}")  # Should be ~22
best_params_clean = {k: v for k, v in best_params.items() 
                     if k != 'scale_pos_weight'}
# Retrain XGBoost
model_final = XGBClassifier(
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1,
    **best_params_clean  # your tuned params from before
)
model_final.fit(X_train_final, y_train)

# Check metrics
y_proba = model_final.predict_proba(X_test_final)[:, 1]
y_pred = (y_proba >= 0.55).astype(int)
print(classification_report(y_test, y_pred))

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

brf = BalancedRandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
brf.fit(X_train, y_train)

y_proba_brf = brf.predict_proba(X_test)[:, 1]
y_pred = (y_proba_brf >= 0.55).astype(int)
print(classification_report(y_test, y_pred))

In [ ]:
from imblearn.ensemble import EasyEnsembleClassifier

ee = EasyEnsembleClassifier(
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)
ee.fit(X_train, y_train)

y_proba_ee = ee.predict_proba(X_test)[:, 1]
y_pred = (y_proba_ee >= 0.55).astype(int)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

auc_roc = roc_auc_score(y_test, y_proba)
auc_pr = average_precision_score(y_test, y_proba)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")

In [ ]:
df.to_csv('different_models_experiment_data.csv')